<a href="https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub pandas

import os
import duckdb
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("✅ Connected")

✅ Connected


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

My baseline rule prioritizes pages that have high impressions but relatively low CTR or poor average position. These pages already receive visibility in search results, so improving their content may increase clicks and overall performance.

### Reason Codes

- LOW_CTR_HIGH_IMPRESSIONS
- LOW_VISIBILITY
- REFRESH_RECOMMENDED

In [ ]:
print("Baseline Rule")

rule = """
Priority Score =
+ High impressions
+ Low CTR
+ Poor average position

Higher score = Higher priority for review
"""

print(rule)

reason_codes = [
    "LOW_CTR_HIGH_IMPRESSIONS",
    "LOW_VISIBILITY",
    "REFRESH_RECOMMENDED"
]

print("Reason Codes:")
for r in reason_codes:
    print("-", r)

Baseline Rule

Priority Score =
+ High impressions
+ Low CTR
+ Poor average position

Higher score = Higher priority for review

Reason Codes:
- LOW_CTR_HIGH_IMPRESSIONS
- LOW_VISIBILITY
- REFRESH_RECOMMENDED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

print("Baseline Rule")

rule = """
Priority Score =
+ High impressions
+ Low CTR
+ Poor average position

Higher score = Higher priority for review
"""

print(rule)

reason_codes = [
    "LOW_CTR_HIGH_IMPRESSIONS",
    "LOW_VISIBILITY",
    "REFRESH_RECOMMENDED"
]

print("Reason Codes:")
for r in reason_codes:
    print("-", r)

In [ ]:
import pandas as pd
import os

# Read sample data
sample = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {fact_daily}
LIMIT 1000
""").df()

# Calculate CTR manually
sample["gsc_ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0, 1)
) * 100

# Baseline score
sample["baseline_score"] = (
    sample["gsc_impressions"] * 0.5
    + (100 - sample["gsc_ctr"]) * 2
    + sample["gsc_avg_position"]
)

sample["action"] = "Review Content"
sample["reason_code"] = "LOW_CTR_HIGH_IMPRESSIONS"

ranked = sample.sort_values(
    by="baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Top 10 Ranked Pages")
display(ranked.head(10))

print("\nCSV saved successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Top 10 Ranked Pages


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,baseline_score,action,reason_code
446,client_9958f0a7ae1df715,content_f94fe855380e150f,303,3,1.811881,0.990099,351.331683,Review Content,LOW_CTR_HIGH_IMPRESSIONS
146,client_9958f0a7ae1df715,content_f94fe855380e150f,266,6,1.812030,2.255639,330.300752,Review Content,LOW_CTR_HIGH_IMPRESSIONS
766,client_9958f0a7ae1df715,content_f94fe855380e150f,240,5,1.808333,2.083333,317.641667,Review Content,LOW_CTR_HIGH_IMPRESSIONS
973,client_9958f0a7ae1df715,content_46c6dc48d36a2ae0,37,0,85.351351,0.000000,303.851351,Review Content,LOW_CTR_HIGH_IMPRESSIONS
852,client_ff644d8251367cbb,content_9884db00882fe43f,20,0,93.100000,0.000000,303.100000,Review Content,LOW_CTR_HIGH_IMPRESSIONS
835,client_ff644d8251367cbb,content_e54a2b04d7baea63,17,0,94.529412,0.000000,303.029412,Review Content,LOW_CTR_HIGH_IMPRESSIONS
953,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,202,4,5.034653,1.980198,302.074257,Review Content,LOW_CTR_HIGH_IMPRESSIONS
204,client_ff644d8251367cbb,content_3c2e782d3c81c503,1,0,101.000000,0.000000,301.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS
259,client_ff644d8251367cbb,content_1d4c3551a46e2967,1,0,100.000000,0.000000,300.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS
501,client_ff644d8251367cbb,content_b241043c0bb3c017,1,0,100.000000,0.000000,300.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS



CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

The following ranked pages were reviewed based on the baseline action score. Each recommendation is interpreted as decision-support rather than a guaranteed outcome. The review explains why each page appears in the ranking and what factors could make the recommendation incorrect.

In [ ]:
top20 = ranked.head(20).copy()

top20["confidence_note"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "The page may already be optimized or influenced by external factors not included in the dataset."
)

display(
    top20[
        [
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
446,content_f94fe855380e150f,351.331683,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
146,content_f94fe855380e150f,330.300752,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
766,content_f94fe855380e150f,317.641667,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
973,content_46c6dc48d36a2ae0,303.851351,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
852,content_9884db00882fe43f,303.100000,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
835,content_e54a2b04d7baea63,303.029412,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
953,content_37b3bafd5f88fdd1,302.074257,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
204,content_3c2e782d3c81c503,301.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
259,content_1d4c3551a46e2967,300.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...
501,content_b241043c0bb3c017,300.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS,Medium,The page may already be optimized or influence...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks and Leakage Check

Some pages may receive a high baseline score because of the selected scoring rule rather than a genuine opportunity for improvement. The baseline uses only observable search-performance signals and does not include future outcomes or label-derived information, helping reduce the risk of data leakage.

In [ ]:
print("Leakage Check")

print("✓ No future performance columns used")
print("✓ No label-derived features used")
print("✓ Baseline uses only observable search signals")

print("\nPossible Weak Picks")

weak = ranked.tail(5)

display(
    weak[
        [
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
        ]
    ]
)

Leakage Check
✓ No future performance columns used
✓ No label-derived features used
✓ Baseline uses only observable search signals

Possible Weak Picks


,content_hash_id,baseline_score,action,reason_code
390,content_0672d8db776419c0,147.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS
325,content_d0d52c7ff7217dac,140.166667,Review Content,LOW_CTR_HIGH_IMPRESSIONS
410,content_5313a91d61b68041,136.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS
729,content_58aaebe6d05bbcf2,110.000000,Review Content,LOW_CTR_HIGH_IMPRESSIONS
73,content_d046e6789ac9ae05,104.500000,Review Content,LOW_CTR_HIGH_IMPRESSIONS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.